In [1]:
import re 
import unicodedata
from collections import Counter

import regex



In [2]:
GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )



### Helper functions and Logic

In [13]:
def pre_tokenize(text):
    """Pre Tokenize the input text based on the GPT2 Regex pattern"""
    return [match.group() for match in GPT2_PATTERN.finditer(text)]

# pre_tokenize("Hello, I am testing this and this ain't working fine.")
# this gives ['Hello', ',', ' I', ' am', ' testing',' this', ' and', ' this', ' ain', "'t", ' working', ' fine','.']

def apply_merge(byte_seq, pair, new_id):
    """
    Apply BPE merge on byte sequence to replace old byte ids with new id for pair if match is found

    If byte_seq = [1, 2, 1, 2]
    pair = [1, 2]
    new_id = 99
    then return [99,99]
    """
    merged = []
    i = 0
    while i < len(byte_seq):
        if i < len(byte_seq) - 1 and byte_seq[i] == pair[0] and byte_seq[i + 1] == pair[1]:
            merged.append(new_id)
            i += 2
        else:
            merged.append(byte_seq[i])
            i += 1
    return merged


class SpecialTokenHandler:
    def __init__(self):
        self.special_tokens = {}
        self.pattern = None

    def add_token(self, token_str, token_id):
        """
        It stores the token string and its ID in self.special_tokens.
        It takes all special tokens currently known.
        It sorts them by length in descending order, so longer tokens are matched before shorter ones.
        It escapes each token with re.escape so characters like . or * are treated literally.
        It joins them into one big alternation pattern, such as:
        token1|token2|token3
        Then it compiles that into a regex and stores it in self.pattern.
        """
        self.special_tokens[token_str] = token_id
        escaped = [re.escape(t) for t in sorted(self.special_tokens.keys(), key=len, reverse=True)]
        self.pattern = re.compile("|".join(escaped))

    def split_with_specials(self, text):
        if not self.pattern:
            return [(text, False)]
        parts = []
        last_end = 0
        for match in self.pattern.finditer(text):
            if match.start() > last_end:
                parts.append((text[last_end:match.start()], False))
            parts.append((match.group(), True))
            last_end = match.end()
        if last_end < len(text):
            parts.append((text[last_end:], False))
        return parts


class ProductionTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.special_handler = SpecialTokenHandler()
        self.next_id = 256

    def normalize(self, text):
        return unicodedata.normalize("NFKC", text)

    def train(self, text, num_merges):
        text = self.normalize(text)
        chunks = pre_tokenize(text)
        chunk_bytes = [list(chunk.encode("utf-8")) for chunk in chunks]

        for i in range(num_merges):
            pairs = Counter()
            for seq in chunk_bytes:
                for j in range(len(seq) - 1):
                    #count number of iterations where x and x+1 are coming together
                    pairs[(seq[j], seq[j + 1])] += 1
            if not pairs:
                break
            ## pick the best pair based on number of occurences
            best = max(pairs, key=pairs.get)

            new_id = self.next_id
            self.next_id += 1
            self.merges[best] = new_id
            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]
            chunk_bytes = [apply_merge(seq, best, new_id) for seq in chunk_bytes]
            merged_display = self.vocab[new_id]
            print(f"Merge {i + 1}: ({best[0]}, {best[1]}) -> {new_id} = {merged_display}")

    def add_special_token(self, token_str):
        token_id = self.next_id
        self.next_id += 1
        self.special_handler.add_token(token_str, token_id)
        self.vocab[token_id] = token_str.encode("utf-8")
        return token_id

    def encode(self, text):
        text = self.normalize(text)
        parts = self.special_handler.split_with_specials(text)
        all_ids = []
        for part_text, is_special in parts:
            if is_special:
                all_ids.append(self.special_handler.special_tokens[part_text])
            else:
                for chunk in pre_tokenize(part_text):
                    byte_seq = list(chunk.encode("utf-8"))
                    for pair, new_id in self.merges.items():
                        byte_seq = apply_merge(byte_seq, pair, new_id)
                    all_ids.extend(byte_seq)
        return all_ids

    def decode(self, ids):
        byte_parts = []
        for token_id in ids:
            if token_id in self.vocab:
                byte_parts.append(self.vocab[token_id])
        return b"".join(byte_parts).decode("utf-8", errors="replace")

    def vocab_size(self):
        return len(self.vocab)

    def get_token_bytes(self, token_id):
        return self.vocab.get(token_id, b"<?>")



## Demos

In [ ]:

def demo_byte_encoding():
    """ Demo for normal byte level encoding to utf-8"""
    print("=" * 60)
    print("Byte-Level Encoding")
    print("=" * 60)

    texts = [
        ("English", "hello"),
        ("Chinese", "你好"),
        ("Japanese", "こんにちは"),
        ("Emoji", "🔥🌍"),
        ("Mixed", "hello你好🔥"),
        ("Code", "def f(x):"),
    ]

    for label, text in texts:
        b = list(text.encode("utf-8"))
        # print("Encoded list",b)
        print(f"{label:10s}: {len(text):2d} chars -> {len(b):2d} bytes -> {b[:16]}{'...' if len(b) > 16 else ''}")
demo_byte_encoding()

Byte-Level Encoding
English   :  5 chars ->  5 bytes -> [104, 101, 108, 108, 111]
Chinese   :  2 chars ->  6 bytes -> [228, 189, 160, 229, 165, 189]
Japanese  :  5 chars -> 15 bytes -> [227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175]
Emoji     :  2 chars ->  8 bytes -> [240, 159, 148, 165, 240, 159, 140, 141]
Mixed     :  8 chars -> 15 bytes -> [104, 101, 108, 108, 111, 228, 189, 160, 229, 165, 189, 240, 159, 148, 165]
Code      :  9 chars ->  9 bytes -> [100, 101, 102, 32, 102, 40, 120, 41, 58]


In [11]:
def demo_pre_tokenization():
    """ Demo for our pre tokenizer step according to GPT2 patteren"""
    print("\n" + "=" * 60)
    print("Pre-Tokenization (GPT-2 Regex)")
    print("=" * 60)

    texts = [
        "Hello, world! Don't stop.",
        "def train(model, data):",
        "The price is $3.14 per unit.",
        "  multiple   spaces   here  ",
    ]

    for text in texts:
        chunks = pre_tokenize(text)
        print(f"\n'{text}'")
        print(f"  -> {chunks}")
demo_pre_tokenization()


Pre-Tokenization (GPT-2 Regex)

'Hello, world! Don't stop.'
  -> ['Hello', ',', ' world', '!', ' Don', "'t", ' stop', '.']

'def train(model, data):'
  -> ['def', ' train', '(', 'model', ',', ' data', '):']

'The price is $3.14 per unit.'
  -> ['The', ' price', ' is', ' $', '3', '.', '14', ' per', ' unit', '.']

'  multiple   spaces   here  '
  -> [' ', ' multiple', '  ', ' spaces', '  ', ' here', '  ']


In [15]:
def demo_full_tokenizer():
    """ Demo for full tokenizer built above after training the tokenizer on corpus below"""
    print("\n" + "=" * 60)
    print("Training Production Tokenizer")
    print("=" * 60)

    corpus = (
        "The quick brown fox jumps over the lazy dog. "
        "The quick brown fox runs through the forest. "
        "Machine learning models process natural language. "
        "Machine learning transforms how we build software. "
        "Deep learning models need large datasets to train. "
        "def train(model, data): return model.fit(data) "
        "def predict(model, x): return model(x) "
        "for i in range(100): print(i) "
    )

    tok = ProductionTokenizer()

    #train on corpus with MAX 50 merges for pairs
    tok.train(corpus, num_merges=50)

    bos_id = tok.add_special_token("<|begin|>")
    eos_id = tok.add_special_token("<|end|>")
    user_id = tok.add_special_token("<|user|>")
    asst_id = tok.add_special_token("<|assistant|>")

    print(f"\nVocab size: {tok.vocab_size()}")
    print(f"Special tokens: <|begin|>={bos_id}, <|end|>={eos_id}, <|user|>={user_id}, <|assistant|>={asst_id}")

    print("\n" + "=" * 60)
    print("Encoding Tests")
    print("=" * 60)

    test_texts = [
        "The quick brown fox.",
        "你好世界 Hello World",
        "🔥🌍🚀",
        "def foo(x): return x + 1",
        "<|begin|><|user|>Hello<|end|>",
        "Machine learning is powerful.",
    ]

    for text in test_texts:
        ids = tok.encode(text)
        decoded = tok.decode(ids)
        raw_bytes = len(text.encode("utf-8"))
        print(f"\nInput:   {text}")
        print(f"IDs:     {ids[:20]}{'...' if len(ids) > 20 else ''}")
        print(f"Tokens:  {len(ids)} (from {raw_bytes} bytes, ratio: {len(ids)/raw_bytes:.2f})")
        print(f"Decoded: {decoded}")
        roundtrip = "PASS" if decoded == text else "FAIL"
        print(f"Round-trip: {roundtrip}")


demo_full_tokenizer()


Training Production Tokenizer
Merge 1: (105, 110) -> 256 = b'in'
Merge 2: (100, 101) -> 257 = b'de'
Merge 3: (32, 116) -> 258 = b' t'
Merge 4: (32, 108) -> 259 = b' l'
Merge 5: (109, 111) -> 260 = b'mo'
Merge 6: (260, 257) -> 261 = b'mode'
Merge 7: (261, 108) -> 262 = b'model'
Merge 8: (102, 111) -> 263 = b'fo'
Merge 9: (114, 101) -> 264 = b're'
Merge 10: (114, 110) -> 265 = b'rn'
Merge 11: (114, 97) -> 266 = b'ra'
Merge 12: (104, 101) -> 267 = b'he'
Merge 13: (114, 111) -> 268 = b'ro'
Merge 14: (32, 263) -> 269 = b' fo'
Merge 15: (32, 262) -> 270 = b' model'
Merge 16: (97, 116) -> 271 = b'at'
Merge 17: (117, 105) -> 272 = b'ui'
Merge 18: (32, 98) -> 273 = b' b'
Merge 19: (259, 97) -> 274 = b' la'
Merge 20: (32, 100) -> 275 = b' d'
Merge 21: (259, 101) -> 276 = b' le'
Merge 22: (276, 97) -> 277 = b' lea'
Merge 23: (277, 265) -> 278 = b' learn'
Merge 24: (278, 256) -> 279 = b' learnin'
Merge 25: (279, 103) -> 280 = b' learning'
Merge 26: (32, 112) -> 281 = b' p'
Merge 27: (103, 101) ->

In [19]:

def demo_tiktoken_comparison():

    """ 
    fertility close to 1 means the text is split into about one token per word
    fertility greater than 1 means the text is broken into more tokens than words, which is common for languages or formats that are harder to segment
    
    """
    try:
        import tiktoken
    except ImportError:
        print("\ntiktoken not installed. Run: pip install tiktoken")
        return

    print("\n" + "=" * 60)
    print("Comparison with tiktoken (GPT-4)")
    print("=" * 60)

    enc = tiktoken.get_encoding("cl100k_base")

    test_paragraph = "Machine learning is powerful. 机器学习很强大。 L'apprentissage automatique est puissant. 🤖💪"

    tokens = enc.encode(test_paragraph)
    pieces = [enc.decode([t]) for t in tokens]

    print(f"\nInput: {test_paragraph}")
    print(f"GPT-4 tokens ({len(tokens)}): {pieces}")

    languages = [
        ("English", "The quick brown fox jumps over the lazy dog."),
        ("Chinese", "快速的棕色狐狸跳过了懒狗。"),
        ("Japanese", "素早い茶色のキツネが怠け者の犬を飛び越えた。"),
        ("Korean", "빠른 갈색 여우가 게으른 개를 뛰어넘었다."),
        ("Code", "def quicksort(arr): return sorted(arr)"),
        ("Emoji", "🎉🎊🎈🎁🎂🎄🎃🎆🎇✨"),
    ]

    print(f"\n{'Language':<10} {'Chars':<6} {'Tokens':<7} {'Fertility':<10}")
    print("-" * 35)
    for label, text in languages:
        toks = enc.encode(text)
        # print(text,toks)
        words = len(text.split())
        fertility = len(toks) / max(words, 1)
        print(f"{label:<10} {len(text):<6} {len(toks):<7} {fertility:<10.2f}")


demo_tiktoken_comparison()


Comparison with tiktoken (GPT-4)

Input: Machine learning is powerful. 机器学习很强大。 L'apprentissage automatique est puissant. 🤖💪
GPT-4 tokens (33): ['Machine', ' learning', ' is', ' powerful', '.', ' ', '机', '器', '学', '�', '�', '�', '�', '�', '�', '大', '。', ' L', "'app", 'rent', 'iss', 'age', ' automat', 'ique', ' est', ' pu', 'issant', '.', ' �', '�', '�', '�', '�']

Language   Chars  Tokens  Fertility 
-----------------------------------
English    44     10      1.11      
Chinese    13     24      24.00     
Japanese   22     32      32.00     
Korean     23     24      4.00      
Code       38     9       2.25      
Emoji      10     29      29.00     
